# Four-Channel CNN-Micro Retraining

This notebook retrains CNN-Micro for four active sEMG channels ordered as `[ECRB, ECRL, FDS, FDP]`. The ground/reference electrode is not a model input.

Runtime recommendation: GPU.

In [ ]:
BRANCH = "codex/rp5-four-channel-retrain"
REPO = "https://github.com/post-stroke-rehab/psr-pipeline.git"
SELECTED_VIEW = "dual"
SELECTED_CONTEXT = 4  # 4 windows spans about 500 ms with 200 ms windows and 50% overlap
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = 60
BATCH_SIZE = 128

In [ ]:
!rm -rf /content/psr-pipeline
!git clone --branch {BRANCH} {REPO} /content/psr-pipeline
%cd /content/psr-pipeline
!pip install -q -r requirements.txt

## Download PhysioMio

If Hugging Face asks for access or authentication, run `huggingface-cli login` in a new cell and rerun this cell.

In [ ]:
!python scripts/download_physiomio.py

## Preprocess Full and Four-Channel Views

The full 64-channel tensors are needed for the teacher during cross-channel distillation. The reduced-channel preprocessing emits left, right, and dual views with identical patient splits.

In [ ]:
!python scripts/run_preprocess.py --fs 2048 --processed-dir datasets/processed/physiomio --force
!python scripts/run_preprocess.py --fs 2048 --processed-dir datasets/processed/physiomio_rp5_4ch --four-channel --force

## Smoke Test

In [ ]:
!python scripts/run_rp5_4ch_experiments.py --stage smoke

## Mapping and Context Selection

In [ ]:
!python scripts/run_rp5_4ch_experiments.py --stage view --epochs 20 --batch-size {BATCH_SIZE} --device cuda
!python scripts/run_rp5_4ch_experiments.py --stage context --selected-view {SELECTED_VIEW} --epochs 20 --batch-size {BATCH_SIZE} --device cuda

## Five-Seed Final Runs

In [ ]:
seed_args = " ".join(map(str, SEEDS))
!python scripts/run_rp5_4ch_experiments.py --stage direct --selected-view {SELECTED_VIEW} --selected-context {SELECTED_CONTEXT} --seeds {seed_args} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --device cuda
!python scripts/run_rp5_4ch_experiments.py --stage distill --selected-view {SELECTED_VIEW} --selected-context {SELECTED_CONTEXT} --seeds {seed_args} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --device cuda

## Optional Transfer Runs

Set `TRANSFER_CHECKPOINT` to a full 64-channel CNN-Micro checkpoint. Transfer uses a concrete anatomical view because first-layer slicing must map from full channels to the four selected active channels.

In [ ]:
TRANSFER_CHECKPOINT = ""  # e.g. /content/psr-pipeline/results/full64_cnn_micro/checkpoint_best.pt
if TRANSFER_CHECKPOINT:
    !python scripts/run_rp5_4ch_experiments.py --stage transfer --selected-view left --selected-context {SELECTED_CONTEXT} --transfer-checkpoint {TRANSFER_CHECKPOINT} --seeds {seed_args} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --device cuda

## Aggregate and Bundle Results

In [ ]:
!python scripts/run_rp5_4ch_experiments.py --stage aggregate
!mkdir -p experiments/rp5_4ch/final
!zip -r /content/rp5_4ch_results.zip experiments/rp5_4ch results/distill_micro_from_cnn_a0.3_t2.0/checkpoint_best.pt

Download `/content/rp5_4ch_results.zip` from the Colab file browser and place its contents back into the local repository.